# 🌊 DeepScan: RT-DETR Fine-Tuning Pipeline
**Smart India Hackathon - Internal Round Cloud Training**

### Instructions for tomorrow:
1. Go to **Runtime > Change runtime type** in the top menu and select **T4 GPU**.
2. Open the folder icon on the left panel.
3. Upload **`dataset_sctd_yolo.zip`** (79MB) from your laptop to Colab.
4. Upload **`best.pt`** (The 63MB RT-DETR model from your `models/stage2_rtdetr_sctd/weights/` folder).
5. Run the cells below one by one!

In [ ]:
# 1. Install the required AI libraries
!pip install ultralytics sahi clearml

In [ ]:
import os

# 2. Check if you uploaded the files
if not os.path.exists('/content/dataset_sctd_yolo.zip'):
    print("⚠️ ERROR: Please upload dataset_sctd_yolo.zip to the left panel!")
elif not os.path.exists('/content/best.pt'):
    print("⚠️ ERROR: Please upload best.pt to the left panel!")
else:
    print("✅ Files detected! Unzipping dataset...")
    !unzip -q /content/dataset_sctd_yolo.zip -d /content/dataset
    print("✅ Dataset Unzipped successfully!")

In [ ]:
# 3. Fix the YAML paths for Google Colab
import glob
import yaml

yaml_files = glob.glob('/content/dataset/**/*.yaml', recursive=True)
if not yaml_files:
    print("⚠️ ERROR: Could not find data.yaml in the dataset!")
else:
    target_yaml = yaml_files[0]
    print(f"Found YAML at: {target_yaml}")
    
    # Rewrite paths to match Colab environment
    with open(target_yaml, 'r') as f:
        data = yaml.safe_load(f)
        
    data['path'] = os.path.dirname(target_yaml)
    
    with open(target_yaml, 'w') as f:
        yaml.dump(data, f)
        
    print("✅ YAML configuration updated for Colab!")
    YAML_PATH = target_yaml

In [ ]:
# 4. FIRE UP THE GPU AND FINE-TUNE!
from ultralytics import RTDETR

# Load the MVP weights you uploaded
print("Loading Pre-trained RT-DETR Foundation Model...")
model = RTDETR('/content/best.pt')

# Fine-tune for 20 epochs (Takes ~2 hours on T4 GPU)
print("🚀 STARTING FINE-TUNING ON T4 GPU...")
results = model.train(
    data=YAML_PATH,
    epochs=20,
    imgsz=640,
    batch=8, # 8 fits perfectly in T4 16GB VRAM
    project='DeepScan_Training',
    name='rtdetr_finetuned',
    device=0 # Force GPU
)
print("✅ TRAINING COMPLETE!")

In [ ]:
# 5. Zip the new weights so you can download them
import shutil
weights_dir = '/content/DeepScan_Training/rtdetr_finetuned/weights'
if os.path.exists(weights_dir):
    shutil.make_archive('/content/FINETUNED_WEIGHTS', 'zip', weights_dir)
    print("🎉 DONE! Download 'FINETUNED_WEIGHTS.zip' from the left panel!")
else:
    print("⚠️ Training didn't finish properly.")